# Actividad 2: Modelos de Predicción y Clasificación

### Grupo: AE_Grupo_6
Integrantes:
* Juan Fernando Gaviria Higuita
* Juan David Ramírez García

## 0. Construcción Automática del DatFrame

Enlace en huggingface del dataset:[HealthRiskPredictor](https://huggingface.co/datasets/infinite-dataset-hub/HealthRiskPredictor)

In [16]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

In [17]:
loader = HFDatasetLoader(
    dataset_name="infinite-dataset-hub/HealthVitalSigns",
    split="train",
    filename="datos/healthvitalSigns.csv",
)

In [18]:
df = loader.load()
df.head()

,idx,timestamp,heart_rate,blood_pressure,oxygen_saturation,body_temperature,label
0,0,2023-01-01T08:00:00Z,72,120/80,98,98.6,normal
1,1,2023-01-01T08:05:00Z,76,121/81,98,98.7,normal
2,2,2023-01-01T08:10:00Z,85,130/85,97,99.1,elevated
3,3,2023-01-01T08:15:00Z,95,135/89,96,100.4,critical
4,4,2023-01-01T08:20:00Z,70,115/75,99,98.4,normal


## 1. Evidencia de preprocesamiento

Revisando estructura del dataFrame

In [19]:
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"Valores nulos totales: {df.isnull().sum().sum()}\n")

print("--- Información de tipos de datos y memoria ---")
df.info()

Filas: 100
Columnas: 7
Filas duplicadas: 0
Valores nulos totales: 0

--- Información de tipos de datos y memoria ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   idx                100 non-null    int64  
 1   timestamp          100 non-null    object 
 2   heart_rate         100 non-null    int64  
 3   blood_pressure     100 non-null    object 
 4   oxygen_saturation  100 non-null    int64  
 5   body_temperature   100 non-null    float64
 6   label              100 non-null    object 
dtypes: float64(1), int64(3), object(3)
memory usage: 5.6+ KB


El dataFrame no tiene datos nulos entonces no haremos limpieza de nulos y de duplicados o imputación de datos.

In [20]:
# Eliminando la columna redundante 'idx' porque es la misma que el índice del DataFrame
df_clean= df.drop(columns=["idx"])
# Verificación de la lista de columnas
print(f"Verificando que la columna 'idx' ha sido eliminada:\n{df_clean.columns}")

Verificando que la columna 'idx' ha sido eliminada:
Index(['timestamp', 'heart_rate', 'blood_pressure', 'oxygen_saturation',
       'body_temperature', 'label'],
      dtype='object')


In [21]:
# Cambiando la columna 'label' por "risk_category
df_clean= df_clean.rename(columns={"label": "risk_category"})
print(f"Verficando cambio de nombre: \n{df_clean.columns}")

Verficando cambio de nombre: 
Index(['timestamp', 'heart_rate', 'blood_pressure', 'oxygen_saturation',
       'body_temperature', 'risk_category'],
      dtype='object')


In [22]:
print("=== COMPARACIÓN DE DATAFRAMES ===")
print(f"Dimensiones Original (df_raw): {df.shape}")
print(f"Dimensiones Limpio   (df_clean): {df_clean.shape}\n")

columnas_eliminadas = set(df.columns) - set(df_clean.columns)
columnas_agregadas = set(df_clean.columns) - set(df.columns)

print(f"Columnas eliminadas: {columnas_eliminadas}")
print(f"Columnas agregadas:   {columnas_agregadas}")

=== COMPARACIÓN DE DATAFRAMES ===
Dimensiones Original (df_raw): (100, 7)
Dimensiones Limpio   (df_clean): (100, 6)

Columnas eliminadas: {'idx', 'label'}
Columnas agregadas:   {'risk_category'}


## 2. Modelo de clasificación e interpretación

La pregunta que queremos responder con este modelo es: **"¿Podemos predecir el nivel de riesgo de salud de un paciente analizando únicamente sus mediciones biométricas básicas, como su ritmo cardíaco, temperatura y presión arterial?"**

Las variables que utilizamos para estructurar esta pregunta se definen así:

Variable Objetivo o "y": La categoría de riesgo de salud (risk_category). Es la etiqueta o diagnóstico final que queremos que el modelo clasifique (por ejemplo, riesgo bajo, medio o alto). Se llama "dependiente" porque asumimos que el estado de salud de la persona es el resultado del comportamiento de sus signos vitales en conjunto.

Variables Predictoras o "X": Las mediciones biométricas (age, heart_rate, oxygen_saturation, body_temperature, systolic_bp, diastolic_bp). Son las pistas o características individuales que le entregamos al modelo matemático (Bosque Aleatorio) para que encuentre patrones y decida a qué categoría de riesgo pertenece el paciente.

In [23]:
# Importar la clase desde tu carpeta clases_po
from clases_po.clasificacion_riesgo import ClasificadorRiesgo

# Inicializar la clase indicando cuál es la columna objetivo inicial
clasificador = ClasificadorRiesgo(target_column="risk_category")

# Preparando y entrenando

X, y = clasificador.preprocesar_datos(df_clean)
clasificador.dividir_datos(X, y)

clasificador.entrenar()

/workspaces/ML_2026Q4/src/clases_po/clasificacion_riesgo.py:40: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  datos[["systolic_bp", "diastolic_bp"]] = datos["blood_pressure"].str.split("/", expand=True).astype(float)
/workspaces/ML_2026Q4/.v

In [24]:
# Prediciendo y evaluando
predicciones_clf = clasificador.predecir()
metricas_clf = clasificador.evaluar()

Accuracy: 0.95
F1-Score (Macro): 0.94

Reporte Detallado:
              precision    recall  f1-score   support

    critical       0.86      1.00      0.92         6
    elevated       1.00      0.83      0.91         6
      normal       1.00      1.00      1.00         8

    accuracy                           0.95        20
   macro avg       0.95      0.94      0.94        20
weighted avg       0.96      0.95      0.95        20



**Accuracy :** El modelo acertó la categoría correcta en el 95% de los pacientes. Sin embargo, esta tasa puede estar un poco inflada por la poca cantidad de datos y podría variar al evaluar casos nuevos.

**Recall :** Responde a: de todos los pacientes realmente graves, ¿cuántos logró detectar el modelo? Un 1.00 significa que los identificó a todos, sin enviar a ningún crítico a casa por error. En el sector salud, esta es la métrica fundamental.

**Precisión :** Responde a: cuando el modelo enciende la alarma de "Paciente Crítico", ¿qué tan confiable es? Un 0.86 muestra que, para asegurar atrapar a todos los graves (Recall de 1.00), el algoritmo se equivoca un 14% de las veces marcando como críticos a pacientes que solo tenían riesgo "elevado".

**F1-Score :** Es el punto de equilibrio entre detectar a todos (Recall) y evitar falsas alarmas (Precision). Un 0.94 indica que, a pesar de la precaución del modelo en casos críticos, mantiene un balance general sobresaliente en todas las categorías.

## Resumen automático del modelo en un archivo .txt

Importando la clase de reportes

In [25]:
# 1. Importar la clase desde tu carpeta clases_po
from clases_po.Reporte_clasificacion import ReporteClasificacion

# 2. Inicializar el generador de reportes
reportador_clf = ReporteClasificacion()

# 3. Definir la ruta donde se guardará el archivo .txt
ruta_reporte = "reportes/reporte_clasificacion.txt"

# 4. Generar y guardar el archivo .txt (pasando los resultados de tu modelo Random Forest)
reportador_clf.guardar_reporte(
    y_test=clasificador.y_test,           # Los datos reales separados por tu clase ClasificadorRiesgo
    predicciones=predicciones_clf,        # Las predicciones que generaste
    ruta=ruta_reporte
)

print(f"Reporte guardado en: {ruta_reporte}\n")

# 5. Imprimir en pantalla para verificar que el texto haya quedado perfecto
reportador_clf.imprimir_reporte(
    y_test=clasificador.y_test, 
    predicciones=predicciones_clf
)

Reporte guardado en: reportes/reporte_clasificacion.txt

REPORTE AUTOMATIZADO DE CLASIFICACIÓN DE RIESGO
Fecha de ejecución: 2026-10-24 14:46:01

1. IDENTIFICACIÓN DEL DATASET Y PREPROCESAMIENTO
--------------------------------------------------
Dataset: Datos biométricos y signos vitales.
Preprocesamiento: Se separó la variable 'blood_pressure' en sistólica y diastólica.
Se descartaron metadatos ('idx', 'timestamp') para dejar únicamente información clínica.

2. MODELO DE CLASIFICACIÓN
--------------------------------------------------
- Objetivo: Clasificar el nivel de riesgo de salud (risk_category).
- Variables Predictoras (X): Edad, ritmo cardíaco, saturación de oxígeno,
  temperatura corporal y presión arterial.
- Algoritmo: Bosque Aleatorio (Random Forest).
- Configuración: Train/Test al 80/20 de forma estratificada, random_state=42.

3. RESULTADOS GLOBALES
--------------------------------------------------
- Accuracy: 0.95
- F1-Score (Macro): 0.94

4. INTERPRETACIÓN CLÍNICA
---